# 02 · Build an agent in 30 minutes

The live-coding agent round, done the way interviewers like to see it. You start with the plain OpenAI SDK: three tools, JSON schemas, the loop, a step budget, error handling and a readable transcript. Then you build the same agent in ten lines with `langchain.agents.create_agent` (built on LangGraph), and add the two extensions they always ask for: memory and human approval before a refund.

**Time:** ~30 min · **Runs:** offline on CPU in < 1 min · **Needs:** [function calling](../08_llm_apps/04_function_calling.ipynb) helps. [The agent loop from scratch](../15_agentic_ai/01_agent_loop_from_scratch.ipynb) goes deeper.

## Why this matters in interviews

- "Build an agent that can use these tools" is the agentic-AI equivalent of FizzBuzz. It shows within minutes whether you understand that **an agent is a loop around a model that can request function calls**.
- Interviewers watch for four things: correct message bookkeeping (every `tool_call_id` answered), a **step budget**, **tool errors fed back as observations** rather than crashing the loop, and a transcript you can debug from.
- The follow-ups (memory, approval for irreversible actions, streaming, parallel calls, observability) are where seniority shows.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `ag02` | What is the core agent loop? |
| `ag04` | What makes a good tool definition? |
| `ag16` | How do you stop an agent looping forever? |
| `ag19` | How do you handle a tool that fails? |
| `ag25` | How do you handle irreversible actions? |
| `ag36` | How would you build a customer support agent that can issue refunds? |
| `ag20` | What is agent memory, and what are its layers? |
| `ag26` | What does an agent observability stack look like? |
| `ag40` | What is the difference between an agent framework and rolling your own? |
| `ag14` | What is human in the loop, and where does it belong? |

## The question, as the interviewer asks it

> *"Build a customer-support agent. It should answer policy questions from our help centre, look up an order's status, and do arithmetic, for example what 15% of an order total is. Use the OpenAI API directly, no framework. Then tell me how you'd take it to production."*

## The first 60 seconds

**Clarifying questions (pick two or three):**

| Ask | Why it matters |
|---|---|
| What do the tools return, and how do they fail? | Failures become observations the model reads. The model needs a clear error, not a stack trace. |
| Are any actions irreversible (refunds, emails)? | Those need a human or policy gate *outside* the model (follow-up 2). |
| Multi-turn or single question? | Decides where the conversation history lives (follow-up 1). |
| Latency and cost budget per conversation? | Sets the step budget (`max_steps`) and the model. |

**The plan, out loud:**

> "An agent is a loop. I send the messages plus JSON schemas of the tools. If the model answers with tool calls, I run each one, append the results as `tool` messages carrying the matching `tool_call_id`, and call the model again. If it answers with text, that's the final answer. Around the loop I add a step budget so it can't spin forever, a wrapper that turns tool exceptions into error messages the model can read, and a transcript of every step. Then I'll show the ten-line framework version, and add memory and an approval gate for refunds."

In [ ]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "llm_setup.py").exists():
        sys.path.insert(0, str(_p)); break
from llm_setup import client, MODEL, EMBED_MODEL, PROVIDER
from llm_setup import langchain_chat_model

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

def draw_pipeline(steps, title=None):
    """Boxes joined by arrows, left to right."""
    fig, ax = plt.subplots(figsize=(min(2.3 * len(steps), 11), 1.7))
    for i, s in enumerate(steps):
        x = i * 2.3
        ax.add_patch(FancyBboxPatch((x, 0), 1.9, 1, boxstyle="round,pad=0.06", fc="#e7f0ff", ec="#3b5bdb"))
        ax.text(x + 0.95, 0.5, s, ha="center", va="center", fontsize=9)
        if i < len(steps) - 1:
            ax.annotate("", xy=(x + 2.28, 0.5), xytext=(x + 1.98, 0.5), arrowprops=dict(arrowstyle="->", lw=1.6))
    ax.set_xlim(-0.2, len(steps) * 2.3); ax.set_ylim(-0.25, 1.25); ax.axis("off")
    if title: ax.set_title(title)
    plt.show()

draw_pipeline(["messages\n+ tool schemas", "model", "tool calls?\nyes: run them", "append\ntool results",
               "loop\n(<= max_steps)", "no calls:\nfinal answer"],
              title="The agent loop: the model proposes, your code executes")

## Packages

```
pip install openai            # the whole plain-SDK agent
pip install langchain langgraph langchain-openai   # only for the framework version
```

| Package | What it gives you | Swap it for |
|---|---|---|
| `openai` | `client.chat.completions.create(..., tools=...)`, which returns `message.tool_calls` | `litellm` (one API for many providers), the `anthropic` SDK, any OpenAI-compatible server (Ollama, vLLM) |
| `json`, `ast` (stdlib) | parse the tool arguments; a **safe** arithmetic evaluator | `pydantic` models to validate arguments |
| `langchain` + `langgraph` | `langchain.agents.create_agent`, the prebuilt agent loop built on LangGraph, plus checkpointers (memory) and interrupts (human approval). The older name is `langgraph.prebuilt.create_react_agent`, deprecated in 1.x | the OpenAI Agents SDK, CrewAI, a hand-written LangGraph graph |
| `langchain-openai` | `ChatOpenAI`, the model object LangGraph drives | `ChatAnthropic`, `ChatOllama` |

## Step 1 · The tools, as plain Python functions

A tiny knowledge base, an orders "database", and a calculator. **Never `eval()` model output.** The calculator walks the syntax tree and allows only arithmetic.

In [ ]:
import ast
import json
import operator
import re
import time

KB = {
    "refunds.md": "Refunds are accepted within 30 days of delivery. Final-sale items cannot be refunded. "
                  "Refunds over $200 need a team lead's approval.",
    "shipping.md": "Standard shipping takes 3 to 5 business days. Express shipping costs $15 and arrives next day.",
    "warranty.md": "Electronics carry a 2-year warranty for manufacturing defects. Accidental damage is not covered.",
}
ORDERS = {
    "ORD-1001": {"status": "delivered", "delivered_on": "2026-09-20", "total": 240.00},
    "ORD-1002": {"status": "shipped", "eta": "2026-10-02", "total": 89.50},
}

_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
        ast.Pow: operator.pow, ast.USub: operator.neg, ast.Mod: operator.mod}

def calculator(expression: str) -> str:
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](ev(node.operand))
        raise ValueError(f"only arithmetic is allowed, got {type(node).__name__}")
    return json.dumps({"expression": expression, "result": round(ev(ast.parse(expression, mode="eval").body), 6)})

def kb_search(query: str) -> str:
    words = set(re.findall(r"[a-z0-9]+", query.lower()))
    ranked = sorted(KB, key=lambda name: -len(words & set(re.findall(r"[a-z0-9]+", KB[name].lower()))))
    return json.dumps([{"source": name, "text": KB[name]} for name in ranked[:2]])

def get_order_status(order_id: str) -> str:
    order = ORDERS.get(order_id.strip().upper())
    if order is None:
        return json.dumps({"error": f"no order with id {order_id!r}; ids look like ORD-1001"})
    return json.dumps({"order_id": order_id, **order})

print(calculator("240 * 0.15"))
print(get_order_status("ORD-1002"))
print(kb_search("refund final sale")[:110], "...")
try:
    calculator("__import__('os').system('echo pwned')")
except ValueError as err:
    print("blocked:", err)

Two design choices to say out loud (`ag04`, `ag07`):

- **Return small JSON, not prose or whole database rows.** The result is the model's *observation*. Short, structured and relevant observations make the next step better and cheaper.
- **Errors say how to recover** ("ids look like ORD-1001"). The model reads that and can retry correctly.

## Step 2 · The JSON schemas

The model never sees your Python. It sees **name + description + parameter schema**, and the descriptions do most of the work of picking the right tool. `strict: True` makes the API guarantee that the arguments match the schema. Strict mode requires every property to be listed in `required` and `additionalProperties: false`.

In [ ]:
def tool_schema(name, description, properties):
    return {"type": "function", "function": {
        "name": name, "description": description, "strict": True,
        "parameters": {"type": "object", "properties": properties,
                       "required": list(properties), "additionalProperties": False}}}

TOOLS = [
    tool_schema("calculator", "Evaluate an arithmetic expression such as '240 * 0.15'. Use it for any maths.",
                {"expression": {"type": "string", "description": "Python-style arithmetic, e.g. (12 + 3) * 2"}}),
    tool_schema("kb_search", "Search the help-centre knowledge base for policy: refunds, shipping, warranty.",
                {"query": {"type": "string", "description": "What to look up, in plain words"}}),
    tool_schema("get_order_status", "Look up the status, dates and total of one customer order by its id.",
                {"order_id": {"type": "string", "description": "An order id like ORD-1001"}}),
]
REGISTRY = {"calculator": calculator, "kb_search": kb_search, "get_order_status": get_order_status}
assert {t["function"]["name"] for t in TOOLS} == set(REGISTRY)
print(json.dumps(TOOLS[2], indent=2))

## Step 3 · Executing a tool call safely

Everything that can go wrong with a tool call becomes a **`{"error": ...}` string returned to the model**. That covers an unknown tool name, malformed JSON arguments, wrong arguments, and the tool raising. The loop never crashes on a tool, and the model gets a chance to recover (`ag19`). Timeouts belong here too, in production.

In [ ]:
from types import SimpleNamespace

def execute_tool(call) -> str:
    name = call.function.name
    fn = REGISTRY.get(name)
    if fn is None:
        return json.dumps({"error": f"unknown tool {name!r}; available: {sorted(REGISTRY)}"})
    try:
        args = json.loads(call.function.arguments or "{}")
    except json.JSONDecodeError as err:
        return json.dumps({"error": f"arguments are not valid JSON: {err}"})
    try:
        return fn(**args)
    except TypeError as err:                        # wrong / missing argument names
        return json.dumps({"error": f"bad arguments for {name}: {err}"})
    except Exception as err:                        # the tool itself failed
        return json.dumps({"error": f"{name} failed: {type(err).__name__}: {err}"})

fake = lambda name, args: SimpleNamespace(function=SimpleNamespace(name=name, arguments=args))
for call in [fake("calculator", '{"expression": "2 ** 10"}'), fake("send_email", "{}"),
             fake("calculator", "{not json"), fake("calculator", '{"expr": "1+1"}'),
             fake("calculator", '{"expression": "1/0"}')]:
    print(f"{call.function.name:12} {call.function.arguments:28} -> {execute_tool(call)}")

## Step 4 · The loop, with a step budget and a transcript

Four details interviewers check:

1. The assistant message that *contains* the tool calls is appended **before** the tool results. Otherwise the API rejects the `tool` messages ("must be a response to a preceding message with tool_calls").
2. **Every** tool call gets a `tool` message with its `tool_call_id`, including the ones that failed.
3. `max_steps` bounds the model calls. When the budget runs out, you return an honest message instead of looping (`ag16`).
4. Each step is recorded, with model latency, tokens and tool calls. That record is your transcript and the seed of observability.

In [ ]:
SYSTEM = ("You are a customer-support agent for an online shop. Use the tools for order data, policy and maths; "
          "never guess an order's status. Keep answers short and cite the policy source file when you use one.")

def run_agent(question, history=None, max_steps=5, tools=TOOLS, executor=execute_tool):
    """Returns (answer, messages, trace). `history` lets a caller carry a conversation across turns."""
    messages = list(history) if history else [{"role": "system", "content": SYSTEM}]
    messages.append({"role": "user", "content": question})
    trace = []
    for step in range(1, max_steps + 1):
        t0 = time.perf_counter()
        resp = client.chat.completions.create(model=MODEL, messages=messages, tools=tools, temperature=0)
        msg = resp.choices[0].message
        trace.append({"step": step, "model_s": time.perf_counter() - t0, "tokens": resp.usage.total_tokens,
                      "tool_calls": [(c.function.name, c.function.arguments) for c in msg.tool_calls or []]})
        messages.append(msg.model_dump(exclude_none=True))       # the assistant turn, tool_calls included
        if not msg.tool_calls:
            return msg.content, messages, trace
        for call in msg.tool_calls:                               # may be several: parallel tool calls
            t1 = time.perf_counter()
            result = executor(call)
            trace[-1].setdefault("tool_s", 0.0)
            trace[-1]["tool_s"] += time.perf_counter() - t1
            messages.append({"role": "tool", "tool_call_id": call.id, "content": result})
    return f"I stopped after {max_steps} steps without a final answer.", messages, trace

def print_transcript(messages):
    for m in messages:
        role = m["role"]
        if role == "system":
            continue
        if role == "assistant" and m.get("tool_calls"):
            for c in m["tool_calls"]:
                print(f"  assistant -> {c['function']['name']}({c['function']['arguments']})")
        elif role == "tool":
            print(f"  tool      <- {m['content'][:110]}")
        else:
            print(f"  {role:9} :  {m.get('content')}")

In [ ]:
QUESTIONS = ["What is 15 percent of 240? Work out 240 * 0.15 for me",
             "Where is my order ORD-1002?",
             "Can I get a refund on a final sale item?",
             "Where is order ORD-9999?"]
for q in QUESTIONS:
    answer, messages, trace = run_agent(q)
    print(f"Q: {q}")
    print_transcript(messages)
    print(f"  steps={len(trace)}  tokens={sum(t['tokens'] for t in trace)}\n")
    tool_ids = {c["id"] for m in messages if m.get("tool_calls") for c in m["tool_calls"]}
    answered = {m["tool_call_id"] for m in messages if m["role"] == "tool"}
    assert tool_ids == answered, "every tool call must get exactly one tool message"

What to notice:

- **The unknown order did not crash anything.** The tool returned an error with a hint, and the model relayed it. With a real model you would typically see *"I couldn't find ORD-9999, could you double-check the id?"*
- Offline, the stand-in picks the tool whose description best matches the question and then quotes the tool result (`Based on the tool results: ...`). `gpt-4.1-mini` does the same job and writes a friendlier sentence. It may also chain steps, for example calling `get_order_status` and then `calculator` on the total. The loop handles that without any change.
- The assert at the bottom is the bookkeeping rule from item 2: every `tool_call_id` is answered exactly once.

The step budget, demonstrated. With `max_steps=1` the model's first reply is a tool call, and the budget ends before it can answer:

In [ ]:
answer, messages, trace = run_agent("Where is my order ORD-1001?", max_steps=1)
print(answer)
assert answer.startswith("I stopped after 1 steps")

## Step 5 · The same agent in 10 lines: `create_agent`

`langchain.agents.create_agent` is the same loop, prebuilt on LangGraph: a model node, a tool node, and an "any tool calls?" edge. The `@tool` decorator builds the JSON schema from the type hints and the docstring. The docstring *is* the description, so write it for the model.

In [ ]:
import warnings
from langchain.agents import create_agent
from langchain_core.tools import tool

@tool
def calculator_tool(expression: str) -> str:
    """Evaluate an arithmetic expression such as '240 * 0.15'. Use it for any maths."""
    return calculator(expression)

@tool
def kb_search_tool(query: str) -> str:
    """Search the help-centre knowledge base for policy: refunds, shipping, warranty."""
    return kb_search(query)

@tool
def get_order_status_tool(order_id: str) -> str:
    """Look up the status, dates and total of one customer order by its id."""
    return get_order_status(order_id)

LC_TOOLS = [calculator_tool, kb_search_tool, get_order_status_tool]
agent = create_agent(langchain_chat_model(temperature=0), LC_TOOLS, system_prompt=SYSTEM)

result = agent.invoke({"messages": [{"role": "user", "content": "Where is my order ORD-1002?"}]})
for m in result["messages"]:
    print(f"  {type(m).__name__:12} {m.content or m.tool_calls}")

**Gotcha worth naming:** tutorials written before LangGraph 1.0 use `from langgraph.prebuilt import create_react_agent`, and you will be asked about it. It still works in 1.x, but it emits a deprecation warning pointing to `create_agent`, and it is scheduled for removal in LangGraph 2.0. The arguments are almost the same, with `prompt=` instead of `system_prompt=`:

In [ ]:
from langgraph.prebuilt import create_react_agent

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    old_agent = create_react_agent(langchain_chat_model(temperature=0), LC_TOOLS, prompt=SYSTEM)
print("warning:", [str(w.message)[:120] for w in caught if "create_react_agent" in str(w.message)])
out = old_agent.invoke({"messages": [("user", "What is 17 * 23?")]})
print([type(m).__name__ for m in out["messages"]], "->", out["messages"][-1].content)

**Framework or hand-rolled (`ag40`)?** Hand-rolled: 40 lines, every byte visible, no churn. The framework gives you checkpointing (memory, resume after a crash), interrupts (human approval), streaming of intermediate steps, and tracing, all for free. A good interview answer: *"I'd prototype with the plain loop to understand it, and use LangGraph once I need durability or human-in-the-loop."*

## The follow-ups they will ask

| Ask | One-line answer |
|---|---|
| **Memory?** | Short-term: keep the message list per conversation and trim it. Long-term: store facts outside and retrieve them. See the code below. |
| **Refunds need a human?** | Gate irreversible tools *in your executor*, not in the prompt. See the code below. |
| **Streaming?** | `stream=True`. Text deltas go to the user as they arrive. Tool-call arguments arrive as fragments you concatenate by `index` before parsing. |
| **Parallel tool calls?** | The model can return several `tool_calls` in one turn. The loop already runs them all, and they can run concurrently with `asyncio.gather`. `parallel_tool_calls=False` turns it off when order matters. |
| **Observability?** | One trace per run, one span per model call and per tool call, carrying latency, tokens, arguments and errors. Use OpenTelemetry, LangSmith or Langfuse. The `trace` list above is the seed. |
| **Cost blow-ups?** | Step budget, token budget per run, per-tool rate limits, and alerts on runs whose step count is an outlier (`ag42`). |

### Follow-up 1 · Memory

**Short-term memory is just the message list.** Keep it per conversation id, pass it back each turn, and trim the oldest turns once it gets long. The system prompt always stays.

In [ ]:
SESSIONS = {}

def chat_turn(session_id, text, keep_last=20):
    history = SESSIONS.get(session_id)
    answer, messages, _ = run_agent(text, history=history)
    system, rest = messages[0], messages[1:]
    SESSIONS[session_id] = [system] + rest[-keep_last:]         # trim, keep the system prompt
    return answer

print("user   : Hi, my name is Sujan.")
print("agent  :", chat_turn("s1", "Hi, my name is Sujan."))
print("user   : What is my name?")
reply = chat_turn("s1", "What is my name?")
print("agent  :", reply)
print("\nnew session, same question ->", chat_turn("s2", "What is my name?"))
assert "Sujan" in reply

Two traps. First, trimming by message count can split an assistant `tool_calls` message from its `tool` replies, and the API rejects a `tool` message whose call was trimmed away. Trim at **turn boundaries**, where a turn is a user message and everything after it. Second, long-term memory ("the customer prefers email") does **not** belong in the chat history. Store it in a table or vector store and retrieve it into the system prompt (`ag20`, `ag30`).

LangGraph gives you the same short-term memory through a **checkpointer**. The state is saved per `thread_id`, and it would survive a restart with a Postgres or SQLite saver:

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

mem_agent = create_agent(langchain_chat_model(temperature=0), LC_TOOLS, system_prompt=SYSTEM,
                         checkpointer=InMemorySaver())
cfg = {"configurable": {"thread_id": "customer-42"}}
mem_agent.invoke({"messages": [{"role": "user", "content": "Hi, my name is Mamitha."}]}, cfg)
out = mem_agent.invoke({"messages": [{"role": "user", "content": "What is my name?"}]}, cfg)
print(out["messages"][-1].content, f"({len(out['messages'])} messages stored in thread customer-42)")
assert "Mamitha" in out["messages"][-1].content

### Follow-up 2 · Human approval before a refund

Add an `issue_refund` tool. The rule to state: **the model proposes, your code disposes.** A prompt that says "ask before refunding" is not a control, because the model can ignore it or be injected into ignoring it. The gate lives in the executor. It checks a policy and, where needed, asks a human before anything irreversible runs (`ag25`, `ag36`).

In [ ]:
REFUNDS_ISSUED = []

def issue_refund(order_id: str, amount: float) -> str:
    REFUNDS_ISSUED.append((order_id, amount))
    return json.dumps({"refunded": amount, "order_id": order_id})

REGISTRY["issue_refund"] = issue_refund
TOOLS_WITH_REFUND = TOOLS + [tool_schema(
    "issue_refund", "Issue a refund of an amount to the customer for an order id. Irreversible.",
    {"order_id": {"type": "string"}, "amount": {"type": "number"}})]
NEEDS_APPROVAL = {"issue_refund"}

def policy_check(name, args):
    """Hard rules first: they apply no matter what the model or the human says."""
    order = ORDERS.get(str(args.get("order_id", "")).upper())
    if order is None:
        return f"unknown order {args.get('order_id')!r}"
    if not 0 < args.get("amount", 0) <= order["total"]:
        return f"amount must be between 0 and the order total {order['total']}"
    return None

def make_gated_executor(ask_human):
    def gated(call):
        name = call.function.name
        if name in NEEDS_APPROVAL:
            args = json.loads(call.function.arguments)
            problem = policy_check(name, args)
            if problem:
                return json.dumps({"error": f"refund blocked by policy: {problem}"})
            if not ask_human(name, args):
                return json.dumps({"error": "a human reviewer declined this refund; tell the customer it is under review"})
        return execute_tool(call)
    return gated

In a notebook the "human" is a function. In production it is a ticket or a Slack button, and the run **pauses** until someone clicks. In LangGraph that is `interrupt()` plus a checkpointer, resumed later with `Command(resume=...)` (see [human-in-the-loop interrupts](../14_langgraph/04_human_in_the_loop_interrupts.ipynb)). Three runs follow: a refund the reviewer approves, one the reviewer declines, and a policy question where the model reaches for the refund tool anyway.

In [ ]:
approvals = []
def reviewer(name, args):                        # stands in for a person: approve small refunds only
    approvals.append((name, args))
    return args["amount"] <= 50

for q in ["Please refund $40 on order ORD-1001, it arrived scratched.",
          "Please refund $90 on order ORD-1001.",
          "What is your refund policy for final sale items?"]:
    answer, messages, _ = run_agent(q, tools=TOOLS_WITH_REFUND, executor=make_gated_executor(reviewer))
    print(f"Q: {q}")
    print_transcript(messages)
    print()

print("refunds actually issued:", REFUNDS_ISSUED)
assert REFUNDS_ISSUED == [("ORD-1001", 40.0)]

Three things in those transcripts:

- **Approved and declined.** The reviewer approved $40 and declined $90, and the declined call came back to the model as an error it can explain to the customer.
- **Parallel calls have no order.** The model asked for `issue_refund` and `get_order_status` *in the same turn*, so "check the order, then refund" was never guaranteed by the model. That is why `policy_check` looks the order up itself.
- **The third question only asked about policy.** Offline, the stand-in matches on the word "refund" and calls `issue_refund` with **invented arguments**. A weak model, or a prompt-injected strong one, does exactly this, and it is why the gate cannot live in the prompt. The policy check blocked it before any human saw it, and only the approved $40 refund was issued. `gpt-4.1-mini` will usually just call `kb_search` here. The gate is for the day it doesn't.

## Where the time and tokens go

Every run above was traced. Plotting tokens per step for one multi-tool conversation makes the cost structure visible. Each extra step re-sends the whole conversation so far, so tokens per call grow step by step (`pd26`).

In [ ]:
_, _, trace_a = run_agent("Where is my order ORD-1001?")
_, _, trace_b = run_agent("Please refund $40 on order ORD-1001.", tools=TOOLS_WITH_REFUND,
                          executor=make_gated_executor(lambda n, a: True))
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
for ax, (label, tr) in zip(axes, [("order status", trace_a), ("refund (4 tools offered)", trace_b)]):
    ax.bar([f"step {t['step']}" for t in tr], [t["tokens"] for t in tr], color=["#4c6ef5", "#f59f00"][:len(tr)])
    ax.set_title(f"{label}: tokens per model call"); ax.set_ylabel("total tokens")
plt.tight_layout(); plt.show()
print("step 1 vs step 2 tokens:", [t["tokens"] for t in trace_a])
assert trace_a[1]["tokens"] > trace_a[0]["tokens"]

## The whole thing as one file: `agent.py`

This is the plain-SDK version you would have on screen at the end of the round, ready to paste into an editor. It needs `pip install openai` and `OPENAI_API_KEY`.

```python
"""agent.py - a customer-support agent: plain OpenAI SDK, 3 tools, a bounded loop.
pip install openai ; env: OPENAI_API_KEY"""
import ast
import json
import operator
import re

from openai import OpenAI

client = OpenAI()
MODEL = "gpt-4.1-mini"
SYSTEM = ("You are a customer-support agent for an online shop. Use the tools for order data, policy and maths; "
          "never guess an order's status. Keep answers short and cite the policy source file when you use one.")

KB = {"refunds.md": "Refunds are accepted within 30 days of delivery. Final-sale items cannot be refunded.",
      "shipping.md": "Standard shipping takes 3 to 5 business days. Express shipping costs $15.",
      "warranty.md": "Electronics carry a 2-year warranty for manufacturing defects."}
ORDERS = {"ORD-1001": {"status": "delivered", "total": 240.00},
          "ORD-1002": {"status": "shipped", "eta": "2026-10-02", "total": 89.50}}

# ---- tools: plain functions that return small JSON strings
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
        ast.Pow: operator.pow, ast.USub: operator.neg, ast.Mod: operator.mod}

def calculator(expression: str) -> str:
    def ev(node):                                  # arithmetic only - never eval() model output
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](ev(node.operand))
        raise ValueError("only arithmetic is allowed")
    return json.dumps({"result": ev(ast.parse(expression, mode="eval").body)})

def kb_search(query: str) -> str:
    words = set(re.findall(r"[a-z0-9]+", query.lower()))
    ranked = sorted(KB, key=lambda n: -len(words & set(re.findall(r"[a-z0-9]+", KB[n].lower()))))
    return json.dumps([{"source": n, "text": KB[n]} for n in ranked[:2]])

def get_order_status(order_id: str) -> str:
    order = ORDERS.get(order_id.strip().upper())
    if order is None:
        return json.dumps({"error": f"no order with id {order_id!r}; ids look like ORD-1001"})
    return json.dumps({"order_id": order_id, **order})

# ---- schemas: what the model actually sees
def tool_schema(name, description, properties):
    return {"type": "function", "function": {
        "name": name, "description": description, "strict": True,
        "parameters": {"type": "object", "properties": properties,
                       "required": list(properties), "additionalProperties": False}}}

TOOLS = [
    tool_schema("calculator", "Evaluate an arithmetic expression such as '240 * 0.15'.",
                {"expression": {"type": "string"}}),
    tool_schema("kb_search", "Search the help-centre knowledge base for policy: refunds, shipping, warranty.",
                {"query": {"type": "string"}}),
    tool_schema("get_order_status", "Look up the status and total of one customer order by its id.",
                {"order_id": {"type": "string"}}),
]
REGISTRY = {"calculator": calculator, "kb_search": kb_search, "get_order_status": get_order_status}

def execute_tool(call) -> str:
    fn = REGISTRY.get(call.function.name)
    if fn is None:
        return json.dumps({"error": f"unknown tool {call.function.name!r}"})
    try:
        return fn(**json.loads(call.function.arguments or "{}"))
    except Exception as err:                       # errors become observations, not crashes
        return json.dumps({"error": f"{type(err).__name__}: {err}"})

# ---- the loop
def run_agent(question, history=None, max_steps=5):
    messages = list(history) if history else [{"role": "system", "content": SYSTEM}]
    messages.append({"role": "user", "content": question})
    for _ in range(max_steps):
        msg = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS,
                                             temperature=0).choices[0].message
        messages.append(msg.model_dump(exclude_none=True))    # assistant turn first
        if not msg.tool_calls:
            return msg.content, messages
        for call in msg.tool_calls:                           # answer every tool_call_id
            messages.append({"role": "tool", "tool_call_id": call.id, "content": execute_tool(call)})
    return f"I stopped after {max_steps} steps without a final answer.", messages

if __name__ == "__main__":
    history = None
    while (q := input("you> ").strip()):
        answer, history = run_agent(q, history)
        print("agent>", answer)
```

## How to explain the key lines

| Line | What to say |
|---|---|
| `tools=TOOLS` | "The model sees only names, descriptions and schemas, so the descriptions are the real prompt engineering." |
| `"strict": True` + `additionalProperties: False` | "The API guarantees the arguments parse and match the schema. There is no 'the model forgot a field'." |
| `messages.append(msg.model_dump(exclude_none=True))` | "The assistant turn with its `tool_calls` goes in first. Tool results must follow the message that requested them." |
| `{"role": "tool", "tool_call_id": call.id, ...}` | "One reply per call id, including failures, or the next request is a 400." |
| `except Exception: return {"error": ...}` | "A failing tool is an observation. The model can retry or apologise, and the loop survives." |
| `for _ in range(max_steps)` | "A hard step budget. An agent without one will eventually burn money in a loop." |
| `history` | "Memory is just the message list, carried per conversation and trimmed at turn boundaries." |

## Try it yourself

**1.** Add a fourth tool, `get_shipping_quote(country: str)`, that returns a price for `"US"`, `"CA"` or `"EU"` and an error otherwise. Register it, add the schema, and ask *"How much is shipping to the EU?"*

In [ ]:
# Solution — try it yourself first, then run this
QUOTES = {"US": 5.0, "CA": 8.0, "EU": 12.0}

def get_shipping_quote(country: str) -> str:
    code = country.strip().upper()
    if code not in QUOTES:
        return json.dumps({"error": f"we do not ship to {country!r}; we ship to {sorted(QUOTES)}"})
    return json.dumps({"country": code, "standard_shipping_usd": QUOTES[code]})

REGISTRY["get_shipping_quote"] = get_shipping_quote
TOOLS_PLUS = TOOLS + [tool_schema("get_shipping_quote", "Get the standard shipping price to a country code: US, CA or EU.",
                                  {"country": {"type": "string", "description": "US, CA or EU"}})]
answer, messages, _ = run_agent("How much is shipping to the EU? My country code is EU.", tools=TOOLS_PLUS)
print_transcript(messages)

**2.** Make the loop refuse to repeat itself: if the model requests the *same tool with the same arguments* twice in one run, return an error to the model instead of running the tool again. This is a cheap guard against the classic "agent stuck in a loop" failure.

In [ ]:
# Solution — try it yourself first, then run this
def make_dedup_executor(inner=execute_tool):
    seen = set()
    def dedup(call):
        key = (call.function.name, call.function.arguments)
        if key in seen:
            return json.dumps({"error": "you already made this exact call; use the earlier result or stop"})
        seen.add(key)
        return inner(call)
    return dedup

ex = make_dedup_executor()
first = ex(fake("get_order_status", '{"order_id": "ORD-1002"}'))
second = ex(fake("get_order_status", '{"order_id": "ORD-1002"}'))
print("first :", first)
print("second:", second)
assert "already made" in second

**3.** Run the four `QUESTIONS` again and report, per question, the number of steps, the total tokens and the tools used, as a small table. (This is what a nightly agent regression job prints. See [agent evaluation](../12_evaluation/04_agent_evaluation.ipynb).)

In [ ]:
# Solution — try it yourself first, then run this
print(f"{'question':44} {'steps':>5} {'tokens':>7}  tools")
for q in QUESTIONS:
    _, _, tr = run_agent(q)
    tools_used = [name for t in tr for name, _ in t["tool_calls"]]
    print(f"{q[:44]:44} {len(tr):5d} {sum(t['tokens'] for t in tr):7d}  {tools_used}")

## Say it in an interview

**30 seconds (`ag02`):** "An agent is a loop. I call the model with the conversation and the tool schemas. If it returns tool calls, I execute them in my code, append each result as a `tool` message with the matching id, and call again. When it returns plain text, that's the answer. I bound it with a step budget, turn tool failures into error observations, and log every step. Irreversible tools go through a policy check and a human approval in the executor, never just in the prompt."

**Numbers and facts:** 3–10 tools per agent before selection quality drops (`ag18`). Step budgets are typically 5–15. Each step re-sends the full history, so cost per step grows. `strict: True` requires every property in `required` plus `additionalProperties: false`.

**Traps:**
- Appending tool results without the assistant `tool_calls` message first gives a 400. Skipping a `tool_call_id` also gives a 400.
- `eval()` on model-produced expressions is remote code execution.
- Trimming history mid-turn orphans tool messages.
- "The prompt tells it to ask first" is not an approval mechanism.
- Old tutorials use `langgraph.prebuilt.create_react_agent`, which is deprecated in LangGraph 1.x in favour of `langchain.agents.create_agent`.

**Follow-ups:** memory (message list per session, checkpointer, long-term store); approval (policy check, then a human, with a pause and resume via `interrupt`); streaming (concatenate tool-argument deltas by `index`); parallel calls (`asyncio.gather`, or `parallel_tool_calls=False`); observability (a trace per run and a span per step).

## Next

- Deeper: [agent loop from scratch](../15_agentic_ai/01_agent_loop_from_scratch.ipynb) · [tool design and function schemas](../15_agentic_ai/02_tool_design_and_function_schemas.ipynb) · [ReAct and plan-and-execute](../15_agentic_ai/03_react_and_plan_and_execute.ipynb) · [agent memory](../15_agentic_ai/04_agent_memory_short_and_long_term.ipynb) · [guardrails, HITL and failure modes](../15_agentic_ai/06_guardrails_hitl_and_failure_modes.ipynb)
- LangGraph: [ReAct tool-calling agent](../14_langgraph/05_react_tool_calling_agent.ipynb) · [checkpointing and memory](../14_langgraph/03_checkpointing_memory_and_time_travel.ipynb) · [human-in-the-loop interrupts](../14_langgraph/04_human_in_the_loop_interrupts.ipynb)
- Other rounds: [RAG skeleton](01_rag_skeleton.ipynb) · [chat API with FastAPI](03_chat_api_with_fastapi.ipynb)
- Theory: [agentic_ai_flow course](../../agentic_ai_flow/index.html) · [interview bank](../../ai_interview_prep/index.html)